In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.optim.swa_utils import AveragedModel

## Configuration

In [ ]:
# ── Chemins ─────────────────────────────────────────────────────────────────
CSV_PATH = "../data/dataset_selection.csv"

MODEL_PATHS = {
    "ConvNeXt-Tiny SWA": "convnext/convnext_tiny_swa.pth",
    "DenseNet-169":      "densenet/densenet169.pth",
    "EfficientNetV2-S":  "efficientnet/efficientnetv2s.pth",
    "ResNet-18":         "resnet/resnet18_best.pth",
}

# Poids d'ensemble proportionnels aux accuracies individuelles (test set sans TTA)
MODEL_WEIGHTS = {
    "ConvNeXt-Tiny SWA": 0.8692,
    "DenseNet-169":      0.8692,
    "EfficientNetV2-S":  0.8564,
    "ResNet-18":         0.8551,
}

# ── TTA ─────────────────────────────────────────────────────────────────────
N_TTA = 5        # nombre de passes TTA (1 = pas de TTA, 5 = recommandé)
IMG_SIZE = 224
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

for name, path in MODEL_PATHS.items():
    status = "✓" if os.path.exists(path) else "✗ MANQUANT"
    print(f"  {status}  {name} → {path}")

## Données — même split que l'entraînement

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["path", "label"]).reset_index(drop=True)
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)

le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])
num_classes = len(le.classes_)
CLASS_NAMES = list(le.classes_)
print("Classes:", CLASS_NAMES)

# Split identique à tous les notebooks d'entraînement (random_state=42)
_, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["y"])
test_df, _ = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["y"])
test_df = test_df.reset_index(drop=True)

print(f"Test set : {len(test_df)} images ({len(test_df)//num_classes} par classe)")

## Utilitaires — crop, Dataset

In [ ]:
def crop_black_border_pil(img: Image.Image, thr=10, pad=10):
    arr = np.array(img)
    mask = arr.mean(axis=2) > thr
    if not mask.any():
        return img
    ys, xs = np.where(mask)
    y0 = max(0, ys.min() - pad);  y1 = min(arr.shape[0] - 1, ys.max() + pad)
    x0 = max(0, xs.min() - pad);  x1 = min(arr.shape[1] - 1, xs.max() + pad)
    return img.crop((x0, y0, x1 + 1, y1 + 1))


class FundusDataset(Dataset):
    def __init__(self, dataframe, transform=None, do_crop=True):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.do_crop = do_crop

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.do_crop:
            img = crop_black_border_pil(img)
        if self.transform:
            img = self.transform(img)
        return img, int(row["y"])

## Transforms TTA

5 passes identiques aux augmentations d'entraînement : original, H-flip, V-flip, rotation +15°, ColorJitter (brightness/contrast/saturation).

In [ ]:
_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

TTA_TRANSFORMS = [
    # 1 — original
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(), _norm,
    ]),
    # 2 — flip horizontal (p=0.5 à l'entraînement)
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(), _norm,
    ]),
    # 3 — flip vertical (p=0.2 à l'entraînement)
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(), _norm,
    ]),
    # 4 — ColorJitter identique à l'entraînement
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(), _norm,
    ]),
    # 5 — rotation +15° (degrees=15 à l'entraînement)
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(15, 15)),
        transforms.ToTensor(), _norm,
    ]),
]

print(f"{N_TTA} passes TTA configurées")

In [ ]:
sample_path = test_df.iloc[0]["path"]
img_pil = crop_black_border_pil(Image.open(sample_path).convert("RGB"))

titles = ["1 — Original", "2 — Flip horizontal", "3 — Flip vertical",
          "4 — ColorJitter", "5 — Rotation +15°"]

to_pil = transforms.ToPILImage()
_denorm = transforms.Compose([
    transforms.Normalize(mean=[0,0,0], std=[1/0.229, 1/0.224, 1/0.225]),
    transforms.Normalize(mean=[-0.485, -0.456, -0.406], std=[1,1,1]),
])

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, tfm, title in zip(axes, TTA_TRANSFORMS, titles):
    ax.imshow(to_pil(_denorm(tfm(img_pil))))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle(os.path.basename(sample_path), fontsize=11)
plt.tight_layout()
plt.show()

## Fonction de prédiction TTA

In [ ]:
def predict_tta(model, dataframe, device, n_tta=N_TTA):
    """Retourne (y_true, y_pred, proba_array) avec moyenne sur n_tta passes."""
    model.eval()
    all_probs = []

    for tfm in TTA_TRANSFORMS[:n_tta]:
        ds = FundusDataset(dataframe, transform=tfm, do_crop=True)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=4, pin_memory=True)
        pass_probs = []
        with torch.no_grad():
            for x, _ in loader:
                probs = torch.softmax(model(x.to(device)), dim=1)
                pass_probs.append(probs.cpu().numpy())
        all_probs.append(np.concatenate(pass_probs, axis=0))

    proba = np.mean(all_probs, axis=0)  # moyenne des passes
    y_true = dataframe["y"].values
    y_pred = proba.argmax(axis=1)
    return y_true, y_pred, proba


def plot_cm(cm, class_names, title="Confusion Matrix"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, data, fmt, t in zip(axes, [cm, cm_norm], ["d", ".2f"],
                                 [title, title + " (normalisée)"]):
        im = ax.imshow(data, cmap="Blues")
        ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
        ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
        ax.set_xlabel("Prédit"); ax.set_ylabel("Vrai"); ax.set_title(t)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                val = data[i, j]
                ax.text(j, i, f"{val:.2f}" if fmt == ".2f" else str(val),
                        ha="center", va="center",
                        color="white" if val > data.max() * 0.6 else "black")
    plt.tight_layout(); plt.show()

## Chargement des 4 modèles

In [ ]:
# ── ConvNeXt-Tiny + SWA ──────────────────────────────────────────────────────
_base_tiny = models.convnext_tiny(weights=None)
_base_tiny.classifier[2] = nn.Linear(_base_tiny.classifier[2].in_features, num_classes)
model_convnext_tiny = AveragedModel(_base_tiny)
model_convnext_tiny.load_state_dict(
    torch.load(MODEL_PATHS["ConvNeXt-Tiny SWA"], map_location=device, weights_only=True))
model_convnext_tiny = model_convnext_tiny.to(device).eval()
print("✓ ConvNeXt-Tiny SWA chargé")

# ── DenseNet-169 ─────────────────────────────────────────────────────────────
model_densenet = models.densenet169(weights=None)
model_densenet.classifier = nn.Linear(model_densenet.classifier.in_features, num_classes)
model_densenet.load_state_dict(
    torch.load(MODEL_PATHS["DenseNet-169"], map_location=device, weights_only=True))
model_densenet = model_densenet.to(device).eval()
print("✓ DenseNet-169 chargé")

# ── EfficientNetV2-S ─────────────────────────────────────────────────────────
model_efficientnet = models.efficientnet_v2_s(weights=None)
model_efficientnet.classifier[1] = nn.Linear(model_efficientnet.classifier[1].in_features, num_classes)
model_efficientnet.load_state_dict(
    torch.load(MODEL_PATHS["EfficientNetV2-S"], map_location=device, weights_only=True))
model_efficientnet = model_efficientnet.to(device).eval()
print("✓ EfficientNetV2-S chargé")

# ── ResNet-18 ────────────────────────────────────────────────────────────────
model_resnet = models.resnet18(weights=None)
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, num_classes)
model_resnet.load_state_dict(
    torch.load(MODEL_PATHS["ResNet-18"], map_location=device, weights_only=True))
model_resnet = model_resnet.to(device).eval()
print("✓ ResNet-18 chargé")

## Évaluation individuelle — sans TTA vs TTA

In [ ]:
loaded_models = {
    "ConvNeXt-Tiny SWA": model_convnext_tiny,
    "DenseNet-169":      model_densenet,
    "EfficientNetV2-S":  model_efficientnet,
    "ResNet-18":         model_resnet,
}

results        = {}  # avec TTA
results_no_tta = {}  # sans TTA

for name, model in loaded_models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    # ── Sans TTA ──────────────────────────────────────────────
    y_true, y_pred, proba = predict_tta(model, test_df, device, n_tta=1)
    results_no_tta[name] = (y_true, y_pred, proba)
    acc_no = (y_true == y_pred).mean()
    auc_no = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    print(f"\n[Sans TTA]  Accuracy : {acc_no:.4f}   |   AUC-ROC : {auc_no:.4f}")

    # ── Avec TTA ──────────────────────────────────────────────
    y_true, y_pred, proba = predict_tta(model, test_df, device, n_tta=N_TTA)
    results[name] = (y_true, y_pred, proba)
    acc_ta = (y_true == y_pred).mean()
    auc_ta = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    print(f"[TTA x{N_TTA}]   Accuracy : {acc_ta:.4f}   |   AUC-ROC : {auc_ta:.4f}   (Δ {acc_ta - acc_no:+.4f})")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
    plot_cm(confusion_matrix(y_true, y_pred), CLASS_NAMES, title=f"{name} (TTA x{N_TTA})")

## Ensemble pondéré (soft voting) — sans TTA vs TTA

In [ ]:
total_weight = sum(MODEL_WEIGHTS[n] for n in loaded_models)
y_true_ens = results["ConvNeXt-Tiny SWA"][0]  # identique pour tous

# ── Ensemble sans TTA ────────────────────────────────────────────────────────
ensemble_proba_no_tta = sum(
    (MODEL_WEIGHTS[name] / total_weight) * results_no_tta[name][2]
    for name in loaded_models
)
y_pred_ens_no_tta = ensemble_proba_no_tta.argmax(axis=1)
acc_ens_no_tta = (y_true_ens == y_pred_ens_no_tta).mean()
auc_ens_no_tta = roc_auc_score(y_true_ens, ensemble_proba_no_tta, multi_class="ovr", average="macro")

print(f"{'='*60}")
print(f"  ENSEMBLE sans TTA")
print(f"{'='*60}")
print(f"\nAccuracy : {acc_ens_no_tta:.4f}   |   AUC-ROC macro : {auc_ens_no_tta:.4f}")
print(classification_report(y_true_ens, y_pred_ens_no_tta, target_names=CLASS_NAMES, digits=4))
plot_cm(confusion_matrix(y_true_ens, y_pred_ens_no_tta), CLASS_NAMES,
        title="Ensemble sans TTA — ConvNeXt-Tiny SWA + DenseNet-169 + EfficientNetV2-S + ResNet-18")

# ── Ensemble avec TTA ────────────────────────────────────────────────────────
ensemble_proba = sum(
    (MODEL_WEIGHTS[name] / total_weight) * results[name][2]
    for name in loaded_models
)
y_pred_ens = ensemble_proba.argmax(axis=1)
acc_ens = (y_true_ens == y_pred_ens).mean()
auc_ens = roc_auc_score(y_true_ens, ensemble_proba, multi_class="ovr", average="macro")

print(f"{'='*60}")
print(f"  ENSEMBLE avec TTA x{N_TTA}")
print(f"{'='*60}")
print(f"\nAccuracy : {acc_ens:.4f}   |   AUC-ROC macro : {auc_ens:.4f}")
print(classification_report(y_true_ens, y_pred_ens, target_names=CLASS_NAMES, digits=4))
plot_cm(confusion_matrix(y_true_ens, y_pred_ens), CLASS_NAMES,
        title=f"Ensemble TTA x{N_TTA} — ConvNeXt-Tiny SWA + DenseNet-169 + EfficientNetV2-S + ResNet-18")

## Récapitulatif

## Images mal classées par l'ensemble (TTA)

In [ ]:
misclassified_idx = np.where(y_true_ens != y_pred_ens)[0]
print(f"Images mal classées (ensemble TTA x{N_TTA}) : {len(misclassified_idx)} / {len(test_df)}")
print(f"Accuracy ensemble : {1 - len(misclassified_idx)/len(test_df):.4f}\n")

# Détail par classe
for cls in CLASS_NAMES:
    cls_idx = int(le.transform([cls])[0])
    cls_mask = (y_true_ens == cls_idx)
    wrong = ((y_true_ens != y_pred_ens) & cls_mask).sum()
    total = cls_mask.sum()
    print(f"  {cls:<12} : {wrong:>3} erreurs / {total}  ({wrong/total*100:.1f}% d'erreur)")

# ── Affichage ────────────────────────────────────────────────────────────────
val_tfm = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

n_show = min(len(misclassified_idx), 25)
n_cols = 5
n_rows = (n_show + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, idx in enumerate(misclassified_idx[:n_show]):
    row = test_df.iloc[idx]
    img = crop_black_border_pil(Image.open(row["path"]).convert("RGB"))
    axes[i].imshow(val_tfm(img).permute(1, 2, 0).numpy())
    true_lbl  = CLASS_NAMES[int(row["y"])]
    pred_lbl  = CLASS_NAMES[y_pred_ens[idx]]
    conf      = ensemble_proba[idx, y_pred_ens[idx]]
    axes[i].set_title(f"Vrai : {true_lbl}\nPrédit : {pred_lbl} ({conf:.2f})", fontsize=8, color="red")
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Images mal classées — ensemble TTA x{N_TTA}  ({len(misclassified_idx)} images)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Modèle':<30} {'Sans TTA':<12} {'TTA x'+str(N_TTA):<12} {'Δ TTA':<10} {'AUC (TTA)'}")
print("-" * 80)

for name in loaded_models:
    yt_no, yp_no, p_no = results_no_tta[name]
    yt, yp, p = results[name]
    acc_no = (yt_no == yp_no).mean()
    acc_ta = (yt == yp).mean()
    auc_ta = roc_auc_score(yt, p, multi_class="ovr", average="macro")
    delta = acc_ta - acc_no
    sign = "+" if delta >= 0 else ""
    print(f"{name:<30} {acc_no:<12.4f} {acc_ta:<12.4f} {sign}{delta:<9.4f} {auc_ta:.4f}")

print("-" * 80)
delta_ens = acc_ens - acc_ens_no_tta
sign = "+" if delta_ens >= 0 else ""
print(f"{'ENSEMBLE':<30} {acc_ens_no_tta:<12.4f} {acc_ens:<12.4f} {sign}{delta_ens:<9.4f} {auc_ens:.4f}")